# Vector Database Setup for Team 3 Hackathon

This notebook sets up a complete vector database solution using Databricks Vector Search. It includes:

* Delta table creation with Change Data Feed
* Vector Search endpoint configuration
* Delta Sync Index with automatic embedding generation
* Sample document insertion and similarity search examples

**Prerequisites**: Ensure you have permissions to create catalogs, schemas, and Vector Search endpoints.

In [0]:
# Configuration
catalog_name = "main"
schema_name = "team3_hackathon"
table_name = "hackathon_documents"
endpoint_name = "team3_hackathon_endpoint"
index_name = "hackathon_docs_index"

# Full table reference
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
full_index_name = f"{catalog_name}.{schema_name}.{index_name}"

print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")
print(f"Table: {full_table_name}")
print(f"Index: {full_index_name}")
print(f"Endpoint: {endpoint_name}")

## Step 1: Create Catalog and Schema

First, we'll create the schema to organize our vector database assets.

In [0]:
# Create schema (catalog 'main' should already exist)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"✓ Schema {catalog_name}.{schema_name} ready")

## Step 2: Create Delta Table with Change Data Feed

We'll create a Delta table with:
* `id` - Unique document identifier
* `text` - The text content to be embedded
* `metadata` - Additional metadata (JSON format)
* `created_at` - Timestamp for tracking

**Change Data Feed (CDF)** is required for Delta Sync, which enables automatic index updates when data changes.

In [0]:
# Drop table if exists (for clean setup)
spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

# Create Delta table with Change Data Feed enabled
spark.sql(f"""
CREATE TABLE {full_table_name} (
  id STRING NOT NULL,
  text STRING NOT NULL,
  metadata STRING,
  created_at TIMESTAMP
) 
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"✓ Table {full_table_name} created with Change Data Feed enabled")

## Step 3: Create Vector Search Endpoint

The Vector Search endpoint provides compute resources for serving the index. This may take a few minutes to provision.

In [0]:
from databricks.vector_search.client import VectorSearchClient

# Initialize Vector Search client
vsc = VectorSearchClient()

# Create endpoint (this will take a few minutes on first run)
try:
    vsc.create_endpoint(
        name=endpoint_name,
        endpoint_type="STANDARD"
    )
    print(f"✓ Endpoint {endpoint_name} creation initiated...")
except Exception as e:
    if "RESOURCE_ALREADY_EXISTS" in str(e):
        print(f"✓ Endpoint {endpoint_name} already exists")
    else:
        raise e

# Wait for endpoint to be ready
import time
print("Waiting for endpoint to be ready...")
while True:
    endpoint = vsc.get_endpoint(endpoint_name)
    status = endpoint.get("endpoint_status", {}).get("state")
    if status == "ONLINE":
        print(f"✓ Endpoint {endpoint_name} is ONLINE")
        break
    elif status in ["PROVISIONING", "UPDATING"]:
        print(f"  Status: {status}...")
        time.sleep(30)
    else:
        print(f"  Unexpected status: {status}")
        break

## Step 4: Create Delta Sync Index with Auto-Embedding

This creates a **Delta Sync Index** that:
* Automatically generates embeddings using Databricks Foundation Models
* Syncs automatically when source table changes (via Change Data Feed)
* Uses the `text` column as the source for embeddings
* Supports metadata filtering

The embedding model used is `databricks-gte-large-en` (768 dimensions).

In [0]:
# Drop index if exists (for clean setup)
try:
    vsc.delete_index(full_index_name)
    print(f"Deleted existing index {full_index_name}")
    time.sleep(10)  # Wait for deletion to complete
except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        pass
    else:
        print(f"Note: {e}")

# Create Delta Sync Index with managed embeddings
index = vsc.create_delta_sync_index(
    endpoint_name=endpoint_name,
    index_name=full_index_name,
    source_table_name=full_table_name,
    pipeline_type="TRIGGERED",  # Can be TRIGGERED or CONTINUOUS
    primary_key="id",
    embedding_source_column="text",  # Column to generate embeddings from
    embedding_model_endpoint_name="databricks-gte-large-en"  # Foundation model
)

print(f"✓ Index {full_index_name} created successfully")
print(f"  Embedding model: databricks-gte-large-en")
print(f"  Pipeline type: TRIGGERED")
print(f"  Primary key: id")
print(f"  Embedding source: text column")

## Step 5: Wait for Index to be Ready

The index needs to be provisioned and synced before we can query it.

In [0]:
# Wait for index to be ready
print("Waiting for index to be ready...")
while True:
    index_status = vsc.get_index(full_index_name)
    status = index_status.get("status", {}).get("detailed_state")
    
    if status == "ONLINE_CONTINUOUS" or status == "ONLINE_TRIGGERED_UPDATE":
        print(f"✓ Index is ONLINE and ready to use")
        break
    elif status in ["PROVISIONING", "SYNCING", "ONLINE_NO_PENDING_UPDATE"]:
        print(f"  Status: {status}...")
        time.sleep(20)
    else:
        print(f"  Current status: {status}")
        time.sleep(20)

## Step 6: Insert Sample Documents

Let's add some sample documents to test the vector database. These could be hackathon project ideas, technical documentation, or any text content.

In [0]:
from pyspark.sql import Row
from datetime import datetime
import json

# Sample hackathon documents
sample_docs = [
    {
        "id": "doc_001",
        "text": "Build a real-time recommendation system using machine learning to suggest products based on user behavior and preferences. Implement collaborative filtering and content-based algorithms.",
        "metadata": json.dumps({"category": "AI/ML", "difficulty": "advanced", "team_size": 4}),
        "created_at": datetime.now()
    },
    {
        "id": "doc_002",
        "text": "Create a mobile app for tracking personal carbon footprint with gamification elements. Users can log daily activities and compete with friends to reduce emissions.",
        "metadata": json.dumps({"category": "Sustainability", "difficulty": "intermediate", "team_size": 3}),
        "created_at": datetime.now()
    },
    {
        "id": "doc_003",
        "text": "Develop a blockchain-based supply chain transparency platform that allows consumers to verify product origins and ethical sourcing through QR codes.",
        "metadata": json.dumps({"category": "Blockchain", "difficulty": "advanced", "team_size": 5}),
        "created_at": datetime.now()
    },
    {
        "id": "doc_004",
        "text": "Build an AI-powered code review assistant that provides intelligent suggestions for code improvements, detects security vulnerabilities, and ensures best practices.",
        "metadata": json.dumps({"category": "DevTools", "difficulty": "advanced", "team_size": 4}),
        "created_at": datetime.now()
    },
    {
        "id": "doc_005",
        "text": "Create an interactive data visualization dashboard for analyzing social media trends in real-time using streaming analytics and beautiful charts.",
        "metadata": json.dumps({"category": "Data Analytics", "difficulty": "intermediate", "team_size": 3}),
        "created_at": datetime.now()
    },
    {
        "id": "doc_006",
        "text": "Design a virtual reality training simulator for emergency responders to practice crisis scenarios in a safe, controlled environment with realistic physics.",
        "metadata": json.dumps({"category": "VR/AR", "difficulty": "advanced", "team_size": 5}),
        "created_at": datetime.now()
    }
]

# Convert to DataFrame and insert
df = spark.createDataFrame([Row(**doc) for doc in sample_docs])
df.write.format("delta").mode("append").saveAsTable(full_table_name)

print(f"✓ Inserted {len(sample_docs)} sample documents")
print(f"\nSample documents:")
for doc in sample_docs:
    print(f"  - {doc['id']}: {doc['text'][:60]}...")

## Step 7: Trigger Index Sync

After inserting data, we need to trigger a sync to update the index with the new embeddings.

In [0]:
# Trigger sync for TRIGGERED pipeline type
vsc.get_index(full_index_name).sync()
print("✓ Index sync triggered")

# Wait for sync to complete
print("Waiting for sync to complete...")
time.sleep(10)  # Give it a moment to start

while True:
    index_status = vsc.get_index(full_index_name)
    status = index_status.get("status", {}).get("detailed_state")
    
    if status == "ONLINE_NO_PENDING_UPDATE":
        print(f"✓ Sync completed - index is ready for queries")
        break
    elif status in ["SYNCING", "ONLINE_TRIGGERED_UPDATE"]:
        print(f"  Syncing...")
        time.sleep(10)
    else:
        print(f"  Status: {status}")
        break

## Step 8: Perform Similarity Search

Now we can search for similar documents using natural language queries. The system will:
1. Convert your query to embeddings using the same model
2. Find the most similar documents using vector similarity
3. Return ranked results with similarity scores

In [0]:
# Example similarity search
query = "I want to build something with artificial intelligence and machine learning"

results = vsc.get_index(full_index_name).similarity_search(
    query_text=query,
    columns=["id", "text", "metadata"],
    num_results=3
)

print(f"Query: '{query}'\n")
print(f"Top {len(results['result']['data_array'])} most similar documents:\n")

for i, doc in enumerate(results['result']['data_array'], 1):
    doc_id = doc[0]
    doc_text = doc[1]
    doc_metadata = json.loads(doc[2]) if doc[2] else {}
    score = doc[3] if len(doc) > 3 else "N/A"
    
    print(f"{i}. Document ID: {doc_id}")
    print(f"   Score: {score}")
    print(f"   Text: {doc_text[:150]}...")
    print(f"   Metadata: {doc_metadata}")
    print()

## Additional Search Examples

Try different queries to see how the vector search performs:

In [0]:
# Try different queries
test_queries = [
    "environmental and sustainability projects",
    "blockchain and distributed systems",
    "visualization and analytics tools"
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'\n")
    
    results = vsc.get_index(full_index_name).similarity_search(
        query_text=query,
        columns=["id", "text"],
        num_results=2
    )
    
    for i, doc in enumerate(results['result']['data_array'], 1):
        print(f"{i}. {doc[0]}: {doc[1][:80]}...")

## Next Steps for Your Hackathon

### Adding More Documents
```python
# Add new documents
new_docs = [
    {"id": "doc_007", "text": "Your document text here", "metadata": json.dumps({...}), "created_at": datetime.now()}
]
df = spark.createDataFrame([Row(**doc) for doc in new_docs])
df.write.format("delta").mode("append").saveAsTable(full_table_name)

# Trigger sync
vsc.get_index(full_index_name).sync()
```

### Filtering with Metadata
```python
# Search with metadata filters
results = vsc.get_index(full_index_name).similarity_search(
    query_text="your query",
    filters={"category": "AI/ML"},
    num_results=5
)
```

### Integration into Your App
* Use the `similarity_search()` API in your application code
* Consider caching frequently queried results
* Monitor index sync status for data freshness
* Scale the endpoint based on query volume

**Good luck with your hackathon! 🚀**